In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
import xgboost

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
NumPy: 2.3.5
Pandas: 2.3.3
Scikit-learn: 1.7.2
XGBoost: 3.4.1


In [2]:
from pathlib import Path
import os

PROJECT_DIR = Path.cwd()

# Find the AirGuard project root
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

print("Project directory:", PROJECT_DIR)
print("Data directory exists:", (PROJECT_DIR / "data").exists())
print("Models directory exists:", (PROJECT_DIR / "models").exists())

print("\nRaw data files:")
for file in (PROJECT_DIR / "data" / "raw").glob("*"):
    print(file.name)

print("\nProcessed data files:")
for file in (PROJECT_DIR / "data" / "processed").glob("*"):
    print(file.name)

print("\nSaved model files:")
for file in (PROJECT_DIR / "models").glob("*"):
    print(file.name)

Project directory: C:\Users\Chaithanya Reddy\Projects\Airguard
Data directory exists: True
Models directory exists: True

Raw data files:
vijayawada_co_jan_mar_2024.csv
vijayawada_no2_jan_mar_2024.csv
vijayawada_o3_jan_mar_2024.csv
vijayawada_pm10_jan_mar_2024.csv
vijayawada_pm25_2024.csv
vijayawada_pm25_jan_mar_2024.csv
vijayawada_so2_jan_mar_2024.csv

Processed data files:
vijayawada_air_quality_clean.csv
vijayawada_features.csv
vijayawada_pm25_clean.csv

Saved model files:
airguard_feature_config.pkl
airguard_xgboost_pm25.pkl


In [3]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / "data" / "processed" / "vijayawada_features.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

Dataset shape: (5800, 28)

Column names:
['collected_at', 'station_id', 'PM10', 'NO2', 'SO2', 'CO', 'PM2.5', 'target_pm25', 'next_time', 'valid_target', 'pm25_lag_1h', 'pm25_lag_2h', 'pm25_lag_3h', 'pm25_lag_6h', 'pm25_lag_12h', 'pm25_lag_24h', 'pm25_rolling_mean_3h', 'pm25_rolling_mean_6h', 'pm25_rolling_mean_12h', 'pm25_rolling_mean_24h', 'hour', 'day_of_week', 'month', 'is_weekend', 'PM10_lag_1h', 'NO2_lag_1h', 'SO2_lag_1h', 'CO_lag_1h']

First 5 rows:


,collected_at,station_id,PM10,NO2,SO2,CO,PM2.5,target_pm25,next_time,valid_target,...,pm25_rolling_mean_12h,pm25_rolling_mean_24h,hour,day_of_week,month,is_weekend,PM10_lag_1h,NO2_lag_1h,SO2_lag_1h,CO_lag_1h
0,2024-01-02 00:00:00,site_5685,27.19,15.30,1.23,0.39,20.80,19.13,2024-01-02 01:00:00,True,...,43.799167,39.040833,0,1,1,0,43.87,13.82,1.23,0.39
1,2024-01-02 01:00:00,site_5685,23.76,15.32,1.23,0.39,19.13,18.60,2024-01-02 02:00:00,True,...,41.703333,38.275833,1,1,1,0,27.19,15.30,1.23,0.39
2,2024-01-02 02:00:00,site_5685,22.66,14.42,1.23,0.39,18.60,19.66,2024-01-02 03:00:00,True,...,39.425833,37.634583,2,1,1,0,23.76,15.32,1.23,0.39
3,2024-01-02 03:00:00,site_5685,24.84,14.39,1.23,0.39,19.66,19.60,2024-01-02 04:00:00,True,...,37.124167,37.365417,3,1,1,0,22.66,14.42,1.23,0.39
4,2024-01-02 04:00:00,site_5685,24.71,14.23,1.23,0.39,19.60,18.09,2024-01-02 05:00:00,True,...,34.614167,37.277500,4,1,1,0,24.84,14.39,1.23,0.39



Data types:
collected_at              object
station_id                object
PM10                     float64
NO2                      float64
SO2                      float64
CO                       float64
PM2.5                    float64
target_pm25              float64
next_time                 object
valid_target                bool
pm25_lag_1h              float64
pm25_lag_2h              float64
pm25_lag_3h              float64
pm25_lag_6h              float64
pm25_lag_12h             float64
pm25_lag_24h             float64
pm25_rolling_mean_3h     float64
pm25_rolling_mean_6h     float64
pm25_rolling_mean_12h    float64
pm25_rolling_mean_24h    float64
hour                       int64
day_of_week                int64
month                      int64
is_weekend                 int64
PM10_lag_1h              float64
NO2_lag_1h               float64
SO2_lag_1h               float64
CO_lag_1h                float64
dtype: object

Missing values:
collected_at             0
stati

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = (
    PROJECT_DIR / "data" / "processed" / "vijayawada_features.csv"
)

df = pd.read_csv(DATA_PATH)

# Parse timestamps
df["collected_at"] = pd.to_datetime(df["collected_at"])
df["next_time"] = pd.to_datetime(df["next_time"])

# Keep only rows with valid next-hour targets
df = df[df["valid_target"] == True].copy()

# Chronological split
train_df = df[df["collected_at"] < "2024-02-01"].copy()

valid_df = df[
    (df["collected_at"] >= "2024-02-01") &
    (df["collected_at"] < "2024-03-01")
].copy()

test_df = df[
    (df["collected_at"] >= "2024-03-01") &
    (df["collected_at"] < "2024-04-01")
].copy()

# Features used by the model
feature_cols = [
    "PM10", "NO2", "SO2", "CO", "PM2.5",
    "pm25_lag_1h", "pm25_lag_2h", "pm25_lag_3h",
    "pm25_lag_6h", "pm25_lag_12h", "pm25_lag_24h",
    "pm25_rolling_mean_3h", "pm25_rolling_mean_6h",
    "pm25_rolling_mean_12h", "pm25_rolling_mean_24h",
    "hour", "day_of_week", "month", "is_weekend",
    "PM10_lag_1h", "NO2_lag_1h", "SO2_lag_1h", "CO_lag_1h"
]

target_col = "target_pm25"

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Test rows:", len(X_test))
print("Number of features:", len(feature_cols))

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Validation:", X_valid.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

print("\nDate ranges:")
print("Train:", train_df["collected_at"].min(), "to",
      train_df["collected_at"].max())
print("Validation:", valid_df["collected_at"].min(), "to",
      valid_df["collected_at"].max())
print("Test:", test_df["collected_at"].min(), "to",
      test_df["collected_at"].max())

assert X_train.shape[1] == X_valid.shape[1] == X_test.shape[1]
assert list(X_train.columns) == list(X_valid.columns) == list(X_test.columns)
assert not X_train.isna().any().any()
assert not X_valid.isna().any().any()
assert not X_test.isna().any().any()
assert y_train.notna().all() and y_valid.notna().all() and y_test.notna().all()

print("\nAll basic data checks passed.")

Training rows: 1513
Validation rows: 1890
Test rows: 2397
Number of features: 23

Missing values:
Train: 0
Validation: 0
Test: 0

Date ranges:
Train: 2024-01-02 00:00:00 to 2024-01-31 23:00:00
Validation: 2024-02-01 00:00:00 to 2024-02-29 23:00:00
Test: 2024-03-01 00:00:00 to 2024-03-31 22:00:00

All basic data checks passed.


In [5]:
import numpy as np
import pandas as pd

# Use January and February only for tuning.
dev_df = df[df["collected_at"] < "2024-03-01"].copy()

# Sort unique timestamps so stations at the same time stay together.
timestamps = np.array(
    sorted(dev_df["collected_at"].unique())
)

print("Development rows:", len(dev_df))
print("Unique timestamps:", len(timestamps))

# Three expanding-window folds.
n_splits = 3
initial_size = int(len(timestamps) * 0.40)
remaining = len(timestamps) - initial_size
validation_size = remaining // n_splits

folds = []

for i in range(n_splits):
    train_end = initial_size + i * validation_size
    valid_end = (
        initial_size + (i + 1) * validation_size
        if i < n_splits - 1
        else len(timestamps)
    )

    train_times = timestamps[:train_end]
    valid_times = timestamps[train_end:valid_end]

    train_idx = np.flatnonzero(
        dev_df["collected_at"].isin(train_times).to_numpy()
    )
    valid_idx = np.flatnonzero(
        dev_df["collected_at"].isin(valid_times).to_numpy()
    )

    assert len(train_idx) > 0 and len(valid_idx) > 0
    assert train_times.max() < valid_times.min()
    assert set(train_times).isdisjoint(set(valid_times))

    folds.append((train_idx, valid_idx))

    print(f"\nFold {i + 1}")
    print("Training rows:", len(train_idx))
    print("Validation rows:", len(valid_idx))
    print("Train end:", pd.Timestamp(train_times.max()))
    print("Validation start:", pd.Timestamp(valid_times.min()))
    print("Validation end:", pd.Timestamp(valid_times.max()))

print("\nAll chronological fold checks passed.")

Development rows: 3403
Unique timestamps: 1416

Fold 1
Training rows: 1207
Validation rows: 588
Train end: 2024-01-25 13:00:00
Validation start: 2024-01-25 14:00:00
Validation end: 2024-02-06 08:00:00

Fold 2
Training rows: 1795
Validation rows: 739
Train end: 2024-02-06 08:00:00
Validation start: 2024-02-06 09:00:00
Validation end: 2024-02-18 03:00:00

Fold 3
Training rows: 2534
Validation rows: 869
Train end: 2024-02-18 03:00:00
Validation start: 2024-02-18 04:00:00
Validation end: 2024-02-29 23:00:00

All chronological fold checks passed.


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

baseline_results = []

for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    fold_valid = dev_df.iloc[valid_idx]

    y_true = fold_valid["target_pm25"].to_numpy()

    # Persistence: next-hour PM2.5 = current-hour PM2.5
    y_pred = fold_valid["PM2.5"].to_numpy()

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    baseline_results.append({
        "fold": fold_number,
        "train_rows": len(train_idx),
        "validation_rows": len(valid_idx),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

baseline_df = pd.DataFrame(baseline_results)

display(baseline_df.round(4))

print("\nMean metrics across folds:")
print(baseline_df[["MAE", "RMSE", "R2"]].mean().round(4))

print("\nMetric standard deviation across folds:")
print(baseline_df[["MAE", "RMSE", "R2"]].std().round(4))

,fold,train_rows,validation_rows,MAE,RMSE,R2
0,1,1207,588,5.7044,13.8987,0.3469
1,2,1795,739,7.9236,17.1741,0.3676
2,3,2534,869,8.7144,33.1752,-0.0057



Mean metrics across folds:
MAE      7.4475
RMSE    21.4160
R2       0.2363
dtype: float64

Metric standard deviation across folds:
MAE      1.5605
RMSE    10.3146
R2       0.2098
dtype: float64


In [7]:
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_PATH = PROJECT_DIR / "models" / "airguard_xgboost_pm25.pkl"

with open(MODEL_PATH, "rb") as f:
    saved_model = pickle.load(f)

print("Loaded model:", type(saved_model))

Loaded model: <class 'xgboost.sklearn.XGBRegressor'>


In [9]:
# Get the exact feature order expected by the saved model
expected_features = saved_model.get_booster().feature_names

print("Expected feature order:")
print(expected_features)

# Reorder the development dataset to match the saved model
X_model = dev_df[expected_features]

print("\nOur feature order now matches:",
      list(X_model.columns) == expected_features)

assert list(X_model.columns) == expected_features

Expected feature order:
['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'pm25_lag_1h', 'pm25_lag_2h', 'pm25_lag_3h', 'pm25_lag_6h', 'pm25_lag_12h', 'pm25_lag_24h', 'pm25_rolling_mean_3h', 'pm25_rolling_mean_6h', 'pm25_rolling_mean_12h', 'pm25_rolling_mean_24h', 'PM10_lag_1h', 'NO2_lag_1h', 'SO2_lag_1h', 'CO_lag_1h', 'hour', 'day_of_week', 'month', 'is_weekend']

Our feature order now matches: True


In [10]:
xgb_results = []

for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    fold_valid = dev_df.iloc[valid_idx]

    # Use the saved model's exact feature order
    X_fold_valid = fold_valid[expected_features]
    y_fold_valid = fold_valid["target_pm25"].to_numpy()

    predictions = saved_model.predict(X_fold_valid)

    xgb_results.append({
        "fold": fold_number,
        "MAE": mean_absolute_error(y_fold_valid, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_fold_valid, predictions)),
        "R2": r2_score(y_fold_valid, predictions)
    })

xgb_df = pd.DataFrame(xgb_results)

display(xgb_df.round(4))

print("\nMean metrics:")
print(xgb_df[["MAE", "RMSE", "R2"]].mean().round(4))

,fold,MAE,RMSE,R2
0,1,5.0269,11.5070,0.5523
1,2,8.5577,17.3251,0.3565
2,3,10.7743,36.8309,-0.2396



Mean metrics:
MAE      8.1197
RMSE    21.8877
R2       0.2231
dtype: float64


In [11]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

fresh_xgb_results = []

for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    fold_train = dev_df.iloc[train_idx]
    fold_valid = dev_df.iloc[valid_idx]

    X_fold_train = fold_train[expected_features]
    y_fold_train = fold_train["target_pm25"]

    X_fold_valid = fold_valid[expected_features]
    y_fold_valid = fold_valid["target_pm25"]

    # Fixed baseline configuration for reproducibility
    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_fold_train, y_fold_train)
    predictions = model.predict(X_fold_valid)

    fresh_xgb_results.append({
        "fold": fold_number,
        "train_rows": len(fold_train),
        "validation_rows": len(fold_valid),
        "MAE": mean_absolute_error(y_fold_valid, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_fold_valid, predictions)),
        "R2": r2_score(y_fold_valid, predictions)
    })

fresh_xgb_df = pd.DataFrame(fresh_xgb_results)

print("Fresh XGBoost — temporal cross-validation")
display(fresh_xgb_df.round(4))

print("\nMean metrics:")
display(fresh_xgb_df[["MAE", "RMSE", "R2"]].mean().to_frame("mean").round(4))

print("\nStandard deviation:")
display(fresh_xgb_df[["MAE", "RMSE", "R2"]].std().to_frame("std").round(4))

Fresh XGBoost — temporal cross-validation


,fold,train_rows,validation_rows,MAE,RMSE,R2
0,1,1207,588,7.3068,13.4257,0.3906
1,2,1795,739,8.3270,19.0060,0.2255
2,3,2534,869,9.6642,32.2133,0.0518



Mean metrics:


,mean
MAE,8.4327
RMSE,21.5483
R2,0.2226



Standard deviation:


,std
MAE,1.1822
RMSE,9.6484
R2,0.1694


In [12]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from itertools import product
from time import perf_counter

# Controlled parameter combinations
parameter_grid = list(product(
    [2, 4, 6],       # max_depth
    [0.03, 0.05],    # learning_rate
    [3, 8]           # min_child_weight
))

experiment_results = []

for config_id, (depth, rate, child_weight) in enumerate(parameter_grid, start=1):
    fold_scores = []

    for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
        fold_train = dev_df.iloc[train_idx]
        fold_valid = dev_df.iloc[valid_idx]

        X_fold_train = fold_train[expected_features]
        y_fold_train = fold_train["target_pm25"]

        X_fold_valid = fold_valid[expected_features]
        y_fold_valid = fold_valid["target_pm25"]

        model = XGBRegressor(
            n_estimators=300,
            max_depth=depth,
            learning_rate=rate,
            min_child_weight=child_weight,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=3.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

        start = perf_counter()
        model.fit(X_fold_train, y_fold_train)
        predictions = model.predict(X_fold_valid)
        elapsed = perf_counter() - start

        fold_scores.append({
            "fold": fold_number,
            "MAE": mean_absolute_error(y_fold_valid, predictions),
            "RMSE": np.sqrt(mean_squared_error(y_fold_valid, predictions)),
            "R2": r2_score(y_fold_valid, predictions),
            "seconds": elapsed
        })

    experiment_results.append({
        "config_id": config_id,
        "max_depth": depth,
        "learning_rate": rate,
        "min_child_weight": child_weight,
        "mean_MAE": np.mean([x["MAE"] for x in fold_scores]),
        "mean_RMSE": np.mean([x["RMSE"] for x in fold_scores]),
        "mean_R2": np.mean([x["R2"] for x in fold_scores]),
        "std_RMSE": np.std([x["RMSE"] for x in fold_scores]),
        "total_seconds": sum(x["seconds"] for x in fold_scores)
    })

results_df = pd.DataFrame(experiment_results)

results_df = results_df.sort_values(
    ["mean_RMSE", "mean_MAE"]
).reset_index(drop=True)

display(results_df.round(4))

print("Configurations evaluated:", len(results_df))
print("All metric values finite:",
      np.isfinite(results_df[
          ["mean_MAE", "mean_RMSE", "mean_R2", "std_RMSE"]
      ].to_numpy()).all())

,config_id,max_depth,learning_rate,min_child_weight,mean_MAE,mean_RMSE,mean_R2,std_RMSE,total_seconds
0,1,2,0.03,3,7.8942,20.4301,0.3041,7.7026,1.1107
1,3,2,0.05,3,8.0888,20.9501,0.2659,7.7149,1.0557
2,5,4,0.03,3,7.9578,20.9954,0.2677,8.1871,1.2095
3,9,6,0.03,3,8.1663,21.4083,0.2417,8.6128,1.6575
4,7,4,0.05,3,8.2683,21.5361,0.2268,8.1808,1.2610
5,11,6,0.05,3,8.3481,21.8044,0.2096,8.4627,1.6434
6,2,2,0.03,8,8.9289,23.2107,-0.0091,4.7924,1.1357
7,6,4,0.03,8,8.8852,23.3432,-0.0165,4.7871,1.1634
8,10,6,0.03,8,9.1370,23.6321,-0.0415,4.8183,1.7066
9,4,2,0.05,8,9.1767,24.3746,-0.1577,4.1222,0.9774


Configurations evaluated: 12
All metric values finite: True


In [13]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

best_params = {
    "n_estimators": 300,
    "max_depth": 2,
    "learning_rate": 0.03,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.5,
    "reg_lambda": 3.0,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1
}

best_fold_results = []

for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    fold_train = dev_df.iloc[train_idx]
    fold_valid = dev_df.iloc[valid_idx]

    model = XGBRegressor(**best_params)

    model.fit(
        fold_train[expected_features],
        fold_train["target_pm25"]
    )

    pred = model.predict(fold_valid[expected_features])
    actual = fold_valid["target_pm25"]

    best_fold_results.append({
        "fold": fold_number,
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": np.sqrt(mean_squared_error(actual, pred)),
        "R2": r2_score(actual, pred)
    })

best_fold_df = pd.DataFrame(best_fold_results)
display(best_fold_df.round(4))

print("\nAll fold metrics finite:",
      np.isfinite(
          best_fold_df[["MAE", "RMSE", "R2"]].to_numpy()
      ).all())

,fold,MAE,RMSE,R2
0,1,6.6281,12.6337,0.4604
1,2,7.9647,17.7399,0.3253
2,3,9.0897,30.9166,0.1266



All fold metrics finite: True


In [14]:
from pathlib import Path
import pandas as pd

REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

# Save the 12-configuration hyperparameter search
search_results_path = REPORTS_DIR / "week5_hyperparameter_search.csv"
results_df.to_csv(search_results_path, index=False)

# Save the best configuration's fold-level scores
fold_results_path = REPORTS_DIR / "week5_best_model_fold_results.csv"
best_fold_df.to_csv(fold_results_path, index=False)

# Save baseline comparisons
baseline_results_path = REPORTS_DIR / "week5_persistence_baseline.csv"
baseline_df.to_csv(baseline_results_path, index=False)

fresh_xgb_path = REPORTS_DIR / "week5_fresh_xgboost_baseline.csv"
fresh_xgb_df.to_csv(fresh_xgb_path, index=False)

print("Saved files:")
for path in [
    search_results_path,
    fold_results_path,
    baseline_results_path,
    fresh_xgb_path
]:
    print(path.name, "| exists:", path.exists(), "| bytes:", path.stat().st_size)

assert all(path.exists() for path in [
    search_results_path,
    fold_results_path,
    baseline_results_path,
    fresh_xgb_path
])

print("\nAll experiment CSV files saved successfully.")

Saved files:
week5_hyperparameter_search.csv | exists: True | bytes: 1372
week5_best_model_fold_results.csv | exists: True | bytes: 195
week5_persistence_baseline.csv | exists: True | bytes: 250
week5_fresh_xgboost_baseline.csv | exists: True | bytes: 249

All experiment CSV files saved successfully.


In [15]:
from pathlib import Path
import pandas as pd

REPORTS_DIR = PROJECT_DIR / "reports"

files_to_check = [
    "week5_hyperparameter_search.csv",
    "week5_best_model_fold_results.csv",
    "week5_persistence_baseline.csv",
    "week5_fresh_xgboost_baseline.csv"
]

for filename in files_to_check:
    path = REPORTS_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

    if path.exists():
        result = pd.read_csv(path)
        print("  Rows:", len(result))
        print("  Columns:", result.columns.tolist())

week5_hyperparameter_search.csv: FOUND
  Rows: 12
  Columns: ['config_id', 'max_depth', 'learning_rate', 'min_child_weight', 'mean_MAE', 'mean_RMSE', 'mean_R2', 'std_RMSE', 'total_seconds']
week5_best_model_fold_results.csv: FOUND
  Rows: 3
  Columns: ['fold', 'MAE', 'RMSE', 'R2']
week5_persistence_baseline.csv: FOUND
  Rows: 3
  Columns: ['fold', 'train_rows', 'validation_rows', 'MAE', 'RMSE', 'R2']
week5_fresh_xgboost_baseline.csv: FOUND
  Rows: 3
  Columns: ['fold', 'train_rows', 'validation_rows', 'MAE', 'RMSE', 'R2']


In [16]:
import pandas as pd
import numpy as np

# Average metrics across the same three validation folds
comparison_df = pd.DataFrame([
    {
        "Model": "Persistence",
        "Mean_MAE": baseline_df["MAE"].mean(),
        "Mean_RMSE": baseline_df["RMSE"].mean(),
        "Mean_R2": baseline_df["R2"].mean()
    },
    {
        "Model": "Fresh XGBoost baseline",
        "Mean_MAE": fresh_xgb_df["MAE"].mean(),
        "Mean_RMSE": fresh_xgb_df["RMSE"].mean(),
        "Mean_R2": fresh_xgb_df["R2"].mean()
    },
    {
        "Model": "Tuned XGBoost",
        "Mean_MAE": best_fold_df["MAE"].mean(),
        "Mean_RMSE": best_fold_df["RMSE"].mean(),
        "Mean_R2": best_fold_df["R2"].mean()
    }
])

display(comparison_df.round(4))

best_model_name = comparison_df.sort_values(
    ["Mean_RMSE", "Mean_MAE"]
).iloc[0]["Model"]

print("Selected by mean RMSE, then MAE:", best_model_name)

,Model,Mean_MAE,Mean_RMSE,Mean_R2
0,Persistence,7.4475,21.4160,0.2363
1,Fresh XGBoost baseline,8.4327,21.5483,0.2226
2,Tuned XGBoost,7.8942,20.4301,0.3041


Selected by mean RMSE, then MAE: Tuned XGBoost


In [17]:
from xgboost import XGBRegressor
import pickle

best_params = {
    "n_estimators": 300,
    "max_depth": 2,
    "learning_rate": 0.03,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.5,
    "reg_lambda": 3.0,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1
}

# Train on January–February only
X_dev = dev_df[expected_features]
y_dev = dev_df["target_pm25"]

final_model = XGBRegressor(**best_params)
final_model.fit(X_dev, y_dev)

print("Final model trained.")
print("Training rows:", len(X_dev))
print("Features:", len(expected_features))

Final model trained.
Training rows: 3403
Features: 23


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# March test data — not used for tuning
X_test_final = test_df[expected_features]
y_test_final = test_df["target_pm25"]

# Tuned XGBoost predictions
tuned_predictions = final_model.predict(X_test_final)

# Persistence baseline: use current PM2.5 to predict next-hour PM2.5
persistence_predictions = test_df["PM2.5"]

# Calculate evaluation metrics
evaluation_results = pd.DataFrame([
    {
        "Model": "Persistence",
        "MAE": mean_absolute_error(y_test_final, persistence_predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test_final, persistence_predictions)),
        "R2": r2_score(y_test_final, persistence_predictions)
    },
    {
        "Model": "Tuned XGBoost",
        "MAE": mean_absolute_error(y_test_final, tuned_predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test_final, tuned_predictions)),
        "R2": r2_score(y_test_final, tuned_predictions)
    }
])

display(evaluation_results.round(4))

print("March test rows:", len(y_test_final))
print("Predictions contain no missing values:", np.isfinite(tuned_predictions).all())

,Model,MAE,RMSE,R2
0,Persistence,2.4732,4.1057,0.6500
1,Tuned XGBoost,3.0372,4.2316,0.6282


March test rows: 2397
Predictions contain no missing values: True


In [19]:
# Save the final March test evaluation
final_test_path = REPORTS_DIR / "week5_march_test_results.csv"

evaluation_results.to_csv(final_test_path, index=False)

print("Saved:", final_test_path)
print("File exists:", final_test_path.exists())
print("\nFinal March test results:")
display(pd.read_csv(final_test_path).round(4))

Saved: C:\Users\Chaithanya Reddy\Projects\Airguard\reports\week5_march_test_results.csv
File exists: True

Final March test results:


,Model,MAE,RMSE,R2
0,Persistence,2.4732,4.1057,0.6500
1,Tuned XGBoost,3.0372,4.2316,0.6282


In [20]:
import pickle
import json

# Save the tuned model
tuned_model_path = PROJECT_DIR / "models" / "airguard_xgboost_pm25_tuned.pkl"

with open(tuned_model_path, "wb") as file:
    pickle.dump(final_model, file)

# Save the feature configuration and parameters
feature_config = {
    "features": list(expected_features),
    "target": "target_pm25",
    "best_parameters": best_params,
    "training_period": "January-February 2024",
    "test_period": "March 2024",
    "model_type": "XGBoost Regressor"
}

config_path = PROJECT_DIR / "models" / "airguard_xgboost_pm25_tuned_feature_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(feature_config, file, indent=4)

print("Model saved:", tuned_model_path.exists())
print("Feature configuration saved:", config_path.exists())
print("Model path:", tuned_model_path)
print("Config path:", config_path)


Model saved: True
Feature configuration saved: True
Model path: C:\Users\Chaithanya Reddy\Projects\Airguard\models\airguard_xgboost_pm25_tuned.pkl
Config path: C:\Users\Chaithanya Reddy\Projects\Airguard\models\airguard_xgboost_pm25_tuned_feature_config.json


In [21]:
with open(tuned_model_path, "rb") as file:
    reloaded_model = pickle.load(file)

reloaded_predictions = reloaded_model.predict(X_test_final)

print("Reloaded model:", type(reloaded_model).__name__)
print("Prediction count:", len(reloaded_predictions))
print(
    "Predictions match before and after saving:",
    np.allclose(tuned_predictions, reloaded_predictions)
)
print("Predictions are finite:", np.isfinite(reloaded_predictions).all())

Reloaded model: XGBRegressor
Prediction count: 2397
Predictions match before and after saving: True
Predictions are finite: True


In [22]:
from pathlib import Path

required_files = [
    REPORTS_DIR / "week5_hyperparameter_search.csv",
    REPORTS_DIR / "week5_best_model_fold_results.csv",
    REPORTS_DIR / "week5_persistence_baseline.csv",
    REPORTS_DIR / "week5_fresh_xgboost_baseline.csv",
    REPORTS_DIR / "week5_march_test_results.csv",
    PROJECT_DIR / "models" / "airguard_xgboost_pm25_tuned.pkl",
    PROJECT_DIR / "models" / "airguard_xgboost_pm25_tuned_feature_config.json",
]

for path in required_files:
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {path.relative_to(PROJECT_DIR)}")

print("\nAll required files exist:", all(p.is_file() for p in required_files))

FOUND: reports\week5_hyperparameter_search.csv
FOUND: reports\week5_best_model_fold_results.csv
FOUND: reports\week5_persistence_baseline.csv
FOUND: reports\week5_fresh_xgboost_baseline.csv
FOUND: reports\week5_march_test_results.csv
FOUND: models\airguard_xgboost_pm25_tuned.pkl
FOUND: models\airguard_xgboost_pm25_tuned_feature_config.json

All required files exist: True
